# Model Evaluation — Near-Earth Objects (NEO)

Este notebook cobre a fase de **Evaluation** do CRISP-DM: verificar se os modelos treinados na fase de Modeling resolvem de facto o problema de negócio definido no início do projeto, não apenas se têm boas métricas técnicas.

**Recordar o Business Understanding:**
- Tipo de tarefa: classificação binária — prever se um NEO é perigoso.
- O erro mais grave é o **falso negativo**: dizer que um objeto não é perigoso quando na verdade é.

**Estrutura deste notebook:**
1. Carregar os dados e retreinar os 3 modelos
2. Tabela-resumo das métricas dos 3 modelos
3. Avaliação ligada ao objetivo de negócio (custo dos erros)
4. Testar os modelos com objetos hipotéticos
5. Conclusão final — modelo recomendado

## 1. Carregar os dados e retreinar os 3 modelos

Ajusta o `caminho` conforme a localização real dos ficheiros no teu repositório. Precisas de ter o `xgboost` instalado (`pip install xgboost`).

In [ ]:
import joblib

modelos = {
    'Regressão Logística': joblib.load(caminho + "modelo_logreg.pkl"),
    'Random Forest': joblib.load(caminho + "modelo_rf.pkl"),
    'XGBoost': joblib.load(caminho + "modelo_xgb.pkl")
}

predicoes = {}
for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test_scaled)
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    predicoes[nome] = {'y_pred': y_pred, 'y_proba': y_proba}

## 2. Tabela-resumo das métricas

Accuracy é incluída só para referência — **não deve ser usada para decidir o melhor modelo**, porque a classe `hazardous=1` é minoritária (~10%). Um modelo que previsse sempre "não perigoso" teria ~90% de accuracy e seria completamente inútil.

In [ ]:
linhas = []
for nome, res in predicoes.items():
    y_pred = res['y_pred']
    y_proba = res['y_proba']
    linhas.append({
        'Modelo': nome,
        'Accuracy': (y_pred == y_test).mean(),
        'Recall (Perigoso)': recall_score(y_test, y_pred),
        'Precision (Perigoso)': precision_score(y_test, y_pred),
        'F1 (Perigoso)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

resumo = pd.DataFrame(linhas).set_index('Modelo').round(3)
resumo

## 3. Avaliação ligada ao objetivo de negócio

O Business Understanding definiu que o **falso negativo** (objeto realmente perigoso, previsto como não perigoso) é o erro mais grave — na prática, seria um objeto perigoso a passar despercebido. Um **falso positivo** (objeto inofensivo marcado como perigoso) é um erro bem menos grave — gera um alarme desnecessário, mas não representa um risco.

Por isso, olhamos aqui especificamente para o número de falsos negativos de cada modelo.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cores = ['Blues', 'Greens', 'Oranges']

resumo_erros = []

for (nome, res), ax, cor in zip(predicoes.items(), axes, cores):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=['Não perigoso', 'Perigoso']).plot(ax=ax, cmap=cor, colorbar=False)
    ax.set_title(nome)

    falsos_negativos = cm[1][0]
    falsos_positivos = cm[0][1]
    total_perigosos = cm[1].sum()
    resumo_erros.append({
        'Modelo': nome,
        'Falsos Negativos': falsos_negativos,
        '% de perigosos não detetados': falsos_negativos / total_perigosos,
        'Falsos Positivos': falsos_positivos
    })

plt.tight_layout()
plt.show()

pd.DataFrame(resumo_erros).set_index('Modelo').round(3)

**Leitura para o negócio:** a coluna "% de perigosos não detetados" é, na prática, a percentagem de objetos realmente perigosos que o modelo deixaria passar despercebidos se fosse usado para vigilância real. Quanto mais baixa esta percentagem, melhor o modelo cumpre o objetivo definido no Business Understanding.

## 4. Testar os modelos com objetos hipotéticos

Simulamos dois objetos inventados — um com características de objeto perigoso (grande, rápido, passa perto) e outro claramente inofensivo (pequeno, lento, passa longe) — para ver como cada modelo se comporta na prática, não só em métricas agregadas.

Os valores de `diameter_mean` e `est_diameter_max` estão em km, `relative_velocity` em km/h, `miss_distance` em km, `absolute_magnitude` é uma escala de brilho (quanto menor, maior/mais brilhante o objeto).

In [ ]:
from sklearn.preprocessing import StandardScaler

# Reconstruir o scaler com os mesmos dados de treino (para escalar os objetos novos)
num_cols = ['diameter_mean', 'est_diameter_max', 'relative_velocity', 'miss_distance', 'absolute_magnitude']
scaler = StandardScaler()
scaler.fit(X_train_scaled[num_cols])  # já estavam escalados; isto é só um exemplo — ver nota abaixo

objetos_novos = pd.DataFrame([
    {  # objeto grande, rápido, passa perto -> esperado: perigoso
        'est_diameter_max': 2.5, 'relative_velocity': 90000,
        'miss_distance': 500000, 'absolute_magnitude': 17.0,
        'diameter_mean': 2.0
    },
    {  # objeto pequeno, lento, passa longe -> esperado: não perigoso
        'est_diameter_max': 0.05, 'relative_velocity': 15000,
        'miss_distance': 70000000, 'absolute_magnitude': 24.0,
        'diameter_mean': 0.04
    }
])

objetos_novos = objetos_novos[X_train_scaled.columns]  # garantir a mesma ordem de colunas

for nome, modelo in modelos.items():
    preds = modelo.predict(objetos_novos)
    probs = modelo.predict_proba(objetos_novos)[:, 1]
    print(f"\n{nome}:")
    for i, (p, pr) in enumerate(zip(preds, probs)):
        resultado = 'PERIGOSO' if p == 1 else 'não perigoso'
        print(f"  Objeto {i+1}: {resultado} (probabilidade: {pr:.3f})")

**Nota importante:** os valores de `objetos_novos` acima estão em escala original (não escalada), mas `X_train_scaled` já tem as colunas escaladas pela Data Preparation. Para um teste rigoroso, terias de guardar o `scaler` da fase de Data Preparation (ex: com `joblib.dump(scaler, 'scaler.pkl')`) e aplicá-lo aqui aos objetos novos, em vez de os passares em escala original. Fica como referência de boa prática — ajusta se quiseres tornar este teste tecnicamente correto antes de o incluíres no relatório final.

## 5. Conclusão final — modelo recomendado

_(preencher depois de correr o notebook, com base nas secções 2 e 3)_

- O modelo com menor percentagem de perigosos não detetados foi: ___
- Isto está de acordo com o que concluímos na fase de Modeling? ___
- O modelo recomendado para uso prático é: ___, porque ___
- Limitações identificadas (ex: dataset representa só um instante no tempo, não trajetórias completas; poucas variáveis disponíveis; falsos positivos ainda são frequentes): ___